# Train YOLOv11n — 36-Class Digit & Letter Tile Detector

This notebook trains a YOLOv11n object detection model to recognize 36 classes (digits 0-9 + letters A-Z) on physical tiles, then exports the model to ONNX for browser deployment.

**Runtime:** ~4-5 hours on Kaggle P100 GPU (150 epochs, 1248 images).

---

## Prerequisites — Do These Before Running

### 1. Upload the dataset to Kaggle

The dataset must be uploaded as a **Kaggle Dataset** (not embedded in the notebook). The local grouped split (train/valid/test by video source) must be preserved.

On your Mac:
```bash
cd ~/proj/digit-training
zip -r dataset.zip dataset/
```

Then on Kaggle:
1. Go to [kaggle.com/datasets](https://kaggle.com/datasets) → **New Dataset**
2. Name it `digit-tiles-v4` (or any name you like)
3. Upload `dataset.zip`
4. Set visibility to **Private**
5. Click **Create**

### 2. Create this notebook on Kaggle

1. Go to [kaggle.com/code](https://kaggle.com/code) → **New Notebook**
2. Upload this `.ipynb` file (or copy-paste cells)
3. In the right sidebar:
   - **Add Data** → search for your `digit-tiles-v4` dataset → **Add**
   - **Accelerator** → **GPU P100**
4. Click **Run All**

### 3. After training completes

1. Go to the **Output** tab of the notebook
2. Download: `best.pt`, `best.onnx`, `results.png`, `confusion_matrix_normalized.png`

## 1. Setup — Install Ultralytics and Verify GPU

In [ ]:
!pip install ultralytics -q

import torch

if not torch.cuda.is_available():
    raise RuntimeError(
        "No GPU detected! Go to the right sidebar → Accelerator → GPU P100.\n"
        "Then restart the notebook and run again."
    )

print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
print(f"PyTorch: {torch.__version__}")
print(f"CUDA: {torch.version.cuda}")

## 2. Locate and Copy Dataset

Kaggle mounts uploaded datasets at `/kaggle/input/{dataset-name}/`. This cell finds the dataset automatically (regardless of what you named it) and copies it to a writable location.

In [ ]:
import shutil
from pathlib import Path

# Search /kaggle/input/ for a directory containing data.yaml
input_root = Path("/kaggle/input")
dataset_source = None

for candidate in sorted(input_root.rglob("data.yaml")):
    dataset_source = candidate.parent
    break

if dataset_source is None:
    raise FileNotFoundError(
        "Could not find data.yaml in /kaggle/input/.\n"
        "Did you add the dataset? Right sidebar → Add Data → search for your dataset."
    )

print(f"Found dataset at: {dataset_source}")

# Copy to writable location (Kaggle input is read-only)
DATASET_DIR = Path("/kaggle/working/dataset")
if DATASET_DIR.exists():
    shutil.rmtree(DATASET_DIR)
shutil.copytree(dataset_source, DATASET_DIR)

print(f"Copied to: {DATASET_DIR}")

## 3. Fix data.yaml Paths and Verify Dataset

The exported `data.yaml` has relative paths (`./train/images`). YOLO resolves these relative to the yaml file location, which works correctly since we copied the whole directory. But we'll set absolute paths to be safe, and verify the dataset integrity.

In [ ]:
import yaml

data_yaml_path = DATASET_DIR / "data.yaml"

with open(data_yaml_path) as f:
    data_cfg = yaml.safe_load(f)

# Rewrite with absolute paths
data_cfg["train"] = str(DATASET_DIR / "train" / "images")
data_cfg["val"] = str(DATASET_DIR / "valid" / "images")
data_cfg["test"] = str(DATASET_DIR / "test" / "images")

with open(data_yaml_path, "w") as f:
    yaml.dump(data_cfg, f, default_flow_style=False)

print("Updated data.yaml:")
print(data_yaml_path.read_text())

# Verify image and label counts
for split in ("train", "valid", "test"):
    img_dir = DATASET_DIR / split / "images"
    lbl_dir = DATASET_DIR / split / "labels"
    n_img = len(list(img_dir.glob("*.jpg"))) if img_dir.exists() else 0
    n_lbl = len(list(lbl_dir.glob("*.txt"))) if lbl_dir.exists() else 0
    print(f"{split:>5}: {n_img} images, {n_lbl} labels")

assert data_cfg["nc"] == 36, f"Expected 36 classes, got {data_cfg['nc']}"
print(f"\nClasses: {data_cfg['nc']} — {data_cfg['names']}")

## 4. Verify Class Coverage

Confirm all 36 classes appear in each split. The grouped stratified split (done locally by `scripts/split.py`) guarantees this, but let's verify.

In [ ]:
from collections import Counter

CLASS_NAMES = data_cfg["names"]

for split in ("train", "valid", "test"):
    lbl_dir = DATASET_DIR / split / "labels"
    class_counts = Counter()
    for lbl_path in lbl_dir.glob("*.txt"):
        text = lbl_path.read_text().strip()
        if not text:
            continue
        for line in text.splitlines():
            cls_id = int(line.split()[0])
            class_counts[cls_id] += 1

    present = set(class_counts.keys())
    missing = set(range(36)) - present
    print(f"{split}: {len(present)}/36 classes present, {sum(class_counts.values())} total annotations")
    if missing:
        missing_names = [CLASS_NAMES[c] for c in sorted(missing)]
        print(f"  WARNING: Missing classes: {missing_names}")

print("\nClass coverage verified.")

## 5. Train

YOLOv11n, 150 epochs, corrected augmentation:
- `fliplr=0.0` — mirrored characters are invalid (YOLO default 0.5 would corrupt 3, 7, 9, J, Z)
- `degrees=10` — slight rotation for tiles placed at angles
- `cos_lr=True` — cosine annealing for smoother convergence
- `patience=50` — early stopping if validation loss plateaus
- `close_mosaic=15` — disable mosaic for final 15 epochs to improve precision

**Expected time: ~4-5 hours on P100.**

In [ ]:
!yolo detect train \
    data=/kaggle/working/dataset/data.yaml \
    model=yolo11n.pt \
    epochs=150 \
    imgsz=640 \
    device=0 \
    batch=16 \
    patience=50 \
    cos_lr=True \
    fliplr=0.0 \
    flipud=0.0 \
    degrees=10 \
    hsv_h=0.01 \
    hsv_s=0.7 \
    hsv_v=0.5 \
    scale=0.5 \
    translate=0.1 \
    mosaic=1.0 \
    close_mosaic=15 \
    mixup=0.0 \
    copy_paste=0.0

## 6. Verify Training Completed

In [ ]:
# Find the training run directory
runs_dir = Path("runs/detect")
train_dirs = sorted(runs_dir.glob("train*")) if runs_dir.exists() else []

if not train_dirs:
    raise FileNotFoundError("No training run found in runs/detect/. Did training complete?")

TRAIN_DIR = train_dirs[-1]  # Latest run
BEST_PT = TRAIN_DIR / "weights" / "best.pt"

if not BEST_PT.exists():
    raise FileNotFoundError(f"best.pt not found at {BEST_PT}. Training may have failed.")

print(f"Training run: {TRAIN_DIR}")
print(f"Best weights: {BEST_PT} ({BEST_PT.stat().st_size / 1e6:.1f} MB)")
print(f"\nTraining artifacts:")
for f in sorted(TRAIN_DIR.glob("*")):
    if f.is_file():
        print(f"  {f.name}")

## 7. Validate on Test Set

Run validation with plots to generate confusion matrix, PR curves, and per-class metrics.

In [ ]:
!yolo detect val \
    model={BEST_PT} \
    data=/kaggle/working/dataset/data.yaml \
    split=test \
    plots=True \
    conf=0.25

## 8. Display Training Results

In [ ]:
from IPython.display import Image, display

results_png = TRAIN_DIR / "results.png"
if results_png.exists():
    print("Training curves (loss, mAP over epochs):")
    display(Image(filename=str(results_png), width=900))

# Show confusion matrix from val run
val_dirs = sorted(runs_dir.glob("val*"))
if val_dirs:
    val_dir = val_dirs[-1]
    cm = val_dir / "confusion_matrix_normalized.png"
    if cm.exists():
        print("\nNormalized confusion matrix (test set):")
        display(Image(filename=str(cm), width=900))
    pr = val_dir / "PR_curve.png"
    if pr.exists():
        print("\nPrecision-Recall curve:")
        display(Image(filename=str(pr), width=700))

## 9. Export to ONNX

Export for browser deployment via ONNX Runtime Web (WASM). Settings:
- `opset=17` — compatible with ORT Web 1.24
- `half=False` — float32 (WASM doesn't support float16)
- `batch=1` — single frame inference

In [ ]:
!yolo export \
    model={BEST_PT} \
    format=onnx \
    imgsz=640 \
    opset=17 \
    half=False \
    batch=1

## 10. Copy Outputs for Download

Copy weights, ONNX model, and evaluation plots to `/kaggle/working/` so they appear in the **Output** tab.

In [ ]:
output_dir = Path("/kaggle/working")

files_to_copy = [
    (BEST_PT, "best.pt"),
    (BEST_PT.with_suffix(".onnx"), "best.onnx"),
    (TRAIN_DIR / "results.png", "results.png"),
    (TRAIN_DIR / "results.csv", "results.csv"),
    (TRAIN_DIR / "args.yaml", "args.yaml"),
]

# Also copy confusion matrix and PR curves from val run
val_dirs = sorted(runs_dir.glob("val*"))
if val_dirs:
    val_dir = val_dirs[-1]
    for name in ("confusion_matrix_normalized.png", "confusion_matrix.png", "PR_curve.png", "F1_curve.png"):
        p = val_dir / name
        if p.exists():
            files_to_copy.append((p, name))

print("Copying outputs to /kaggle/working/:")
for src, dst_name in files_to_copy:
    dst = output_dir / dst_name
    if src.exists():
        shutil.copy2(src, dst)
        size_mb = dst.stat().st_size / 1e6
        print(f"  {dst_name} ({size_mb:.1f} MB)")
    else:
        print(f"  {dst_name} — NOT FOUND at {src}")

print("\nDone! Go to the Output tab to download these files.")

## Next Steps

After downloading from the Output tab:

1. **Evaluate locally** (M13):
   ```bash
   cd ~/proj/digit-training
   source .venv/bin/activate
   # Copy best.pt to runs/detect/train/weights/
   yolo detect val \
     model=runs/detect/train/weights/best.pt \
     data=dataset/data.yaml \
     plots=True conf=0.25
   ```

2. **Deploy ONNX to TileSight** (M14):
   ```bash
   cp best.onnx ~/proj/superbuilders/public/models/digit-tiles.onnx  # local dir still named "superbuilders"
   ```

3. **Test on device**: Run the app with `?debug=true` and verify detections + latency < 120ms.